In [13]:
# Import necessary libraries
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import pickle
from datetime import datetime
import statsmodels.api as sm
from sklearn.cluster import KMeans
import warnings

warnings.filterwarnings('ignore')

# Import our MarkovKAMA implementation
from markovKama import MarkovKAMA

In [21]:
# Path to Bitcoin data
data_path = 'data/micro/candleData/bitcoin_candles.csv'

# Initialize the MarkovKAMA model
model = MarkovKAMA(data_path=data_path, train_test_split=0.6, init_date='2013-01-01',embargo_pct=0.005)

# Display the first few rows of the data
model.data.head()

Data loaded successfully: 4349 records
Training set: 2609 records
Test set: 1740 records


,open,high,low,close
date,,,,
2013-04-28,135.30,135.30,135.30,135.30
2013-04-29,141.96,141.96,141.96,141.96
2013-04-30,135.30,135.30,135.30,135.30
2013-05-01,117.00,117.00,117.00,117.00
2013-05-02,103.43,103.43,103.43,103.43


In [22]:
# Apply the Markov Switching Regression model
low_var, high_var = model.get_markov(no_regimes=2, return_data=True)

# Plot the probabilities of each regime over time using plotly as a stacked area chart
fig = go.Figure()

# Create a stacked area chart since low_var + high_var = 1
fig.add_trace(
    go.Scatter(
        x=low_var.index, 
        y=low_var, 
        name='Low Volatility',
        line=dict(width=0),
        stackgroup='one',
        fillcolor='rgba(0, 255, 0, 0.7)'
    )
)

fig.add_trace(
    go.Scatter(
        x=high_var.index, 
        y=high_var, 
        name='High Volatility',
        line=dict(width=0),
        stackgroup='one',
        fillcolor='rgba(255, 0, 0, 0.7)'
    )
)

# Calculate the test data start date
test_start_idx = int(len(model.data) * model.train_test_split)
test_start_date = model.data.index[test_start_idx]

# Add vertical line for test data start - using a scatter trace instead of add_vline
fig.add_trace(
    go.Scatter(
        x=[test_start_date, test_start_date],
        y=[0, 1],
        mode='lines',
        line=dict(color='black', width=2, dash='dash'),
        name='Test Data Start',
        showlegend=True
    )
)

# Add annotation for the test data line
fig.add_annotation(
    x=test_start_date,
    y=1,
    text="",
    showarrow=False,
    yshift=10
)

# Update layout
fig.update_layout(
    title='Markov Switching Regime Probabilities for Bitcoin',
    xaxis_title='Date',
    yaxis_title='Probability',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    ),
    height=600,
    width=1000,
    hovermode='x unified'
)

fig.show()

Markov Switching Regression model fitted successfully.
Training data: 2588 points
Embargo data: 21 points
Test data: 1738 points


In [16]:
# Calculate KAMA with default parameters
# There seems to be an issue with the model.get_kama method
# Let's fix the approach by using the correct method call

# First, check if kama is already calculated and stored as an attribute
if hasattr(model, 'kama_values') and isinstance(model.kama_values, pd.Series):
    kama_ = model.kama_values
else:
    # If not, call the method without trying to unpack multiple return values
    model.get_kama(n_window=10, pow1=2, pow2=30, gamma=0.15)
    kama_ = model.kama_values

# Plot BTC price with KAMA using Plotly
# Note: KAMA values are in log scale, so we need to transform them back
price = model.asset_data
# Convert log KAMA values to actual price scale
kama_actual = np.exp(kama_)

fig = go.Figure()

# Add price trace
fig.add_trace(
    go.Scatter(
        x=price.index,
        y=price,
        name='BTC Price',
        line=dict(color='blue', width=1),
        opacity=0.5
    )
)

# Add KAMA trace (converted from log scale)
fig.add_trace(
    go.Scatter(
        x=kama_actual.index,
        y=kama_actual,
        name='KAMA',
        line=dict(color='black', width=2)
    )
)

# Update layout
fig.update_layout(
    title='Bitcoin Price and KAMA',
    xaxis_title='Date',
    yaxis_title='Price (USD)',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    ),
    height=600,
    width=1000,
    hovermode='x unified'
)

fig.show()

KAMA calculation completed successfully.


In [17]:
# Classify market regimes
classes = model.get_classes()

# Count occurrences of each regime
regime_counts = classes['label'].value_counts()
print("Regime Counts:")
print(regime_counts)

# Calculate metrics for each regime segment
metrics = model.calculate_metrics(classes)

# Display regime metrics
print("\nRegime Metrics:")
display(metrics[['label', 'start_date', 'end_date', 'duration', 'slope', 'volatility', 'returns']].tail(10))

Regime classification completed successfully.
Regime Counts:
label
Bullish_Low_Var     2018
Bearish_Low_Var      670
Bullish_High_Var     553
Bearish_High_Var     488
Name: count, dtype: int64

Regime Metrics:


,label,start_date,end_date,duration,slope,volatility,returns
419,Bullish_Low_Var,2024-12-19,2024-12-25,7,0.000654,0.024743,-0.020606
420,Bearish_Low_Var,2024-12-26,2025-01-03,9,0.005647,0.014109,0.041608
421,Bullish_Low_Var,2025-01-04,2025-02-06,34,0.001225,0.023972,-0.017588
422,Bearish_Low_Var,2025-02-07,2025-02-16,10,0.000478,0.012103,-0.004236
423,Bullish_Low_Var,2025-02-17,2025-02-24,8,-0.003656,0.022351,-0.045721
424,Bearish_High_Var,2025-02-25,2025-02-26,2,-0.056265,0.019052,-0.054712
425,Bearish_Low_Var,2025-02-27,2025-02-28,2,-0.003157,0.009018,-0.003152
426,Bearish_High_Var,2025-03-01,2025-03-10,10,-0.011493,0.051839,-0.083960
427,Bearish_High_Var,2025-03-12,2025-03-13,2,-0.033764,0.033081,-0.033201
428,Bearish_Low_Var,2025-03-14,2025-03-18,5,-0.003236,0.023222,-0.014195


In [18]:
print(metrics.groupby('label')['returns'].mean())
print(metrics.groupby('label')['returns'].std())
print(metrics.groupby('label')['returns'].median())
print(metrics.groupby('label')['returns'].skew())



label
Bearish_High_Var    0.008562
Bearish_Low_Var     0.007108
Bullish_High_Var    0.035023
Bullish_Low_Var     0.022448
Name: returns, dtype: float64
label
Bearish_High_Var    0.114183
Bearish_Low_Var     0.028222
Bullish_High_Var    0.157900
Bullish_Low_Var     0.069033
Name: returns, dtype: float64
label
Bearish_High_Var   -0.001701
Bearish_Low_Var     0.001766
Bullish_High_Var    0.004925
Bullish_Low_Var     0.011828
Name: returns, dtype: float64
label
Bearish_High_Var   -0.071422
Bearish_Low_Var     0.710375
Bullish_High_Var    2.165003
Bullish_Low_Var     0.941928
Name: returns, dtype: float64


In [19]:
portfolio = model.implement_trading_strategy(classes, initial_capital=1000)
portfolio

Regime exposure (average position & % of time):
  Bullish_Low_Var: 0.93 (69.4% of time)
  Bullish_High_Var: 0.45 (6.8% of time)
  Bearish_Low_Var: 0.07 (17.6% of time)
  Bearish_High_Var: 0.11 (6.3% of time)


,btc_price,regime,next_open,trading_state,cash,btc_holdings,portfolio_value,trades,trading_costs,cumulative_costs,strategy_returns,btc_returns,buy_hold_value,strategy_normalized,buy_hold_normalized,regime_change,regime_duration
date,,,,,,,,,,,,,,,,,
2023-09-09,25889.0,Bullish_Low_Var,25890.0,0,1000.000000,0.000000,1000.000000,0,0,0,NaN,NaN,1000.000000,1.000000,1.000000,True,44
2023-09-10,25835.0,Bullish_Low_Var,25835.0,1,0.000000,0.038586,996.877752,1,1,1,-0.003122,-0.002086,997.914172,0.996878,0.997914,False,44
2023-09-11,25133.0,Bullish_Low_Var,25163.0,1,0.000000,0.038586,969.790151,0,0,1,-0.027172,-0.027172,970.798409,0.969790,0.970798,False,44
2023-09-12,25867.0,Bullish_Low_Var,25850.0,1,0.000000,0.038586,998.112514,0,0,1,0.029205,0.029205,999.150218,0.998113,0.999150,False,44
2023-09-13,26223.0,Bullish_Low_Var,26222.0,1,0.000000,0.038586,1011.849247,0,0,1,0.013763,0.013763,1012.901232,1.011849,1.012901,False,44
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-03-15,84392.0,Bearish_Low_Var,84345.0,0,2495.752828,0.000000,2495.752828,0,0,56,0.000000,0.005002,3259.762834,2.495753,3.259763,False,5
2025-03-16,82611.0,Bearish_Low_Var,82570.0,0,2495.752828,0.000000,2495.752828,0,0,56,0.000000,-0.021104,3190.969137,2.495753,3.190969,False,5
2025-03-17,84075.0,Bearish_Low_Var,83977.0,0,2495.752828,0.000000,2495.752828,0,0,56,0.000000,0.017722,3247.518251,2.495753,3.247518,False,5


In [20]:
# Generate visualization showing ONLY test set results
fig = model.plot_test_results(portfolio, classes)

# For deployment - using the latest data point's classifi

In [9]:
model.kama_values

date
2015-01-01     5.749361
2015-01-02     5.750825
2015-01-03     5.657983
2015-01-04     5.564290
2015-01-05     5.610277
                ...    
2025-03-15    11.384917
2025-03-16    11.381227
2025-03-17    11.380471
2025-03-18    11.379197
2025-03-19    11.378790
Name: close, Length: 3731, dtype: float64

In [10]:
model.optimize()

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

KAMA calculation completed successfully.              
Regime classification completed successfully.         
KAMA calculation completed successfully.                                        
Regime classification completed successfully.                                   
KAMA calculation completed successfully.                                        
Regime classification completed successfully.                                   
KAMA calculation completed successfully.                                        
Regime classification completed successfully.                                   
KAMA calculation completed successfully.                                        
Regime classification completed successfully.                                   
KAMA calculation completed successfully.                                        
Regime classification completed successfully.                                   
KAMA calculation completed successfully.                                        

{'gamma': 0.1, 'n': 20.0, 'pow1': 9.0, 'pow2': 80.0}

In [11]:
model.get_kama(n_window=15, pow1=5, pow2=55, gamma=0.15)

KAMA calculation completed successfully.
